# 12 — YOLO OBB Card Extraction

Two-part notebook based on the approach from [1vcian/Pokemon-TCGP-Card-Scanner](https://github.com/1vcian/Pokemon-TCGP-Card-Scanner):

### Part A — Pre-trained model (Option 1: run immediately)
Download their YOLO11n-OBB model trained on 10k synthetic Pokemon card images,
run oriented bounding-box detection, perspective-warp the card to a clean crop,
then feed into our existing centering/defect detectors.

### Part B — Custom training dataset (Option 2: better accuracy)
Adapt `trainCreator.py` to composite our own card images onto diverse eBay-style
backgrounds, generate a YOLO OBB dataset, and train a fresh YOLO11n-obb model.

**Why OBB over axis-aligned boxes?**
OBB gives all 4 corners even when the card is tilted — required for a correct
perspective warp that produces a perfectly rectangular crop for grading analysis.

**Prerequisites:** `pip install ultralytics opencv-python-headless`

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from pathlib import Path

# Test images for Part A
TEST_IMAGES = [
    "cards/image0_front.jpeg",
    "cards/image1_front.jpeg",
    "cards/image2_front.jpeg",
]

# Output card crop size (standard Pokemon card 63×88mm → 2.48:3.46 ratio)
CARD_W, CARD_H = 630, 880   # px — crop target for perspective warp

# Detection confidence threshold
CONF_THRESHOLD = 0.25

# Part B — training dataset paths
CARDS_DIR       = Path("datasets/tcg_cards")          # clean official card faces (downloaded in B1)
BACKGROUNDS_DIR = Path("datasets/backgrounds")        # background images (create if needed)
DATASET_OUT     = Path("datasets/yolo_obb_cards")     # generated YOLO dataset
N_TRAIN_IMAGES  = 2000    # synthetic images to generate (use 10000 for full training)
N_TCG_CARDS     = 500     # clean official cards to download for compositing (B1)

# Pre-trained model (Part A) — downloaded automatically by ultralytics
# This is the YOLO11n-obb base; replace with Hub model path once exported
PRETRAINED_BASE = "yolo11n-obb.pt"   # base model for transfer learning

# Their Ultralytics Hub model ID (requires API key for direct download)
HUB_MODEL_ID = "dQfecRsRsXbAKXOXHLHJ"
HUB_API_KEY  = ""   # ← paste your Ultralytics API key here if you have one

VERBOSE = True

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import math, random, shutil, yaml, os
from io import BytesIO
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm
from ultralytics import YOLO

print("✅  Imports OK")

---
## Part A — Pre-trained Model
### A1 — Load model

In [ ]:
def load_obb_model() -> YOLO:
    """
    Load the best available OBB model:
      1. Their custom Hub model (needs API key) → best for Pokemon cards
      2. Cached custom model from previous training (Part B)
      3. YOLO11n-obb base model → general oriented-box detector

    The base model (option 3) is trained on DOTA (aerial objects) so it
    won't recognize "pokemon_card" as a class, but its OBB heads still detect
    rectangular objects. We use it for demo; train your own with Part B.
    """
    # Option 1: Hub model with API key
    if HUB_API_KEY:
        try:
            from ultralytics import hub
            hub.login(HUB_API_KEY)
            model = YOLO(f"https://hub.ultralytics.com/models/{HUB_MODEL_ID}")
            print(f"✅  Loaded Hub model {HUB_MODEL_ID}")
            return model
        except Exception as e:
            print(f"⚠️  Hub load failed: {e}")

    # Option 2: locally trained custom model from Part B
    custom_path = DATASET_OUT / "train" / "weights" / "best.pt"
    if custom_path.exists():
        model = YOLO(str(custom_path))
        print(f"✅  Loaded custom trained model: {custom_path}")
        return model

    # Option 3: base YOLO11n-obb (auto-downloaded ~6 MB)
    model = YOLO(PRETRAINED_BASE)
    print(f"✅  Loaded base {PRETRAINED_BASE} (DOTA pretrained — run Part B for card-specific weights)")
    return model


model = load_obb_model()

### A2 — OBB detection + perspective warp

The OBB result gives 4 corner points `(x1,y1)...(x4,y4)` in pixel coords.
We sort them into TL/TR/BR/BL order and apply `cv2.getPerspectiveTransform`
to warp the card to a clean `CARD_W × CARD_H` rectangle.

In [ ]:
def order_corners(pts: np.ndarray) -> np.ndarray:
    """Sort 4 corner points into [TL, TR, BR, BL] order."""
    pts = pts.reshape(4, 2).astype(np.float32)
    s = pts.sum(axis=1)
    d = np.diff(pts, axis=1)
    ordered = np.zeros((4, 2), dtype=np.float32)
    ordered[0] = pts[np.argmin(s)]   # TL
    ordered[2] = pts[np.argmax(s)]   # BR
    ordered[1] = pts[np.argmin(d)]   # TR
    ordered[3] = pts[np.argmax(d)]   # BL
    return ordered


def perspective_warp(img: np.ndarray, corners: np.ndarray,
                     out_w: int = CARD_W, out_h: int = CARD_H) -> np.ndarray:
    """Perspective-warp the detected card region to a clean rectangle."""
    src = order_corners(corners)
    dst = np.array([[0, 0], [out_w, 0], [out_w, out_h], [0, out_h]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (out_w, out_h), flags=cv2.INTER_LANCZOS4)


def _is_card_like(corners: np.ndarray, img_h: int, img_w: int,
                  min_area_frac: float = 0.05,
                  max_area_frac: float = 0.90) -> bool:
    """
    Accept a detection only if it looks like an actual card boundary:
      - Area ≥ 5% of image     (rejects tiny in-card regions like damage box)
      - Area ≤ 90% of image    (rejects image-boundary false positive)
      - Aspect ratio 1.0–2.2   (portrait or landscape card, allows tilt)
    """
    pts  = corners.reshape(4, 2).astype(np.float32)
    area = cv2.contourArea(pts)
    img_area = img_h * img_w
    if not (min_area_frac * img_area <= area <= max_area_frac * img_area):
        return False
    rect = cv2.minAreaRect(pts)
    rw, rh = rect[1]
    if rw == 0 or rh == 0:
        return False
    aspect = max(rw, rh) / min(rw, rh)
    return 1.0 <= aspect <= 2.2


def _quad_from_mask(mask: np.ndarray) -> Optional[np.ndarray]:
    """convexHull → minAreaRect → boxPoints on largest contour."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt  = max(contours, key=cv2.contourArea)
    hull = cv2.convexHull(cnt)
    rect = cv2.minAreaRect(hull)
    box  = cv2.boxPoints(rect)
    return box.astype(np.float32)


def _tighten_quad_to_edges(img_bgr: np.ndarray, quad: np.ndarray,
                            inward_frac: float = 0.15, outward_frac: float = 0.04,
                            n_samples: int = 60, edge_z: float = 3.5,
                            peak_frac: float = 0.5, method: str = "offset",
                            aspect_tol: float = 0.10, cand_frac: float = 0.35,
                            max_k: int = 4, refine_window: int = 20,
                            debug: bool = False):
    """
    Snap each side of a loose quad to the card's true OUTER edge.

    Per side we build a directional, colour-aware gradient profile (Sobel on
    each LAB channel, projected onto the side normal, averaged across n_samples
    points ALONG the side) and locate the card border as the FIRST rise above
    the background floor (a low percentile of the on-image profile) — ignoring
    the stronger interior holo / text gradients further in.

    method:
      "offset"  — ONE perpendicular offset per side (fast). Assumes the loose
                  side is ~parallel to the card edge, which holds after
                  minAreaRect / Hough for in-plane ROTATION.
      "linefit" — per-along-side peak + robust line fit (cv2.fitLine, Huber), so
                  the 4 sides may be NON-PARALLEL. Handles PERSPECTIVE-skewed
                  (trapezoidal) cards. Falls back to "offset" for any side with
                  too few inlier points.
      "aspect"  — like "offset", but choose among MULTIPLE candidate edges per
                  side the combination whose quad ASPECT RATIO is closest to the
                  card's ~1.4 (CARD_ASPECT_TARGET), preferring strong edges.
                  Rejects a strong gradient spike that is background (stand /
                  shadow) instead of the real card border.

    Tuning (peak_frac is the primary knob):
      peak_frac    — fraction of (peak - floor) the first rise must clear.
                     LOWER -> catches a weaker / further-out edge; HIGHER ->
                     demands a stronger step (may skip the true edge).
      inward_frac  — hard cap on how far inward a side may move (x quad diagonal).
      edge_z       — legacy; unused (kept for call compatibility).

    method="aspect" knobs:
      aspect_tol    — allowed deviation of quad aspect from ~1.4 (fraction).
      cand_frac     — peak height (vs floor) for a side to offer a candidate.
      max_k         — candidate edges considered per side (search ~ (k+1)^4).
      refine_window — +/- px the final uniformity-snap may move each side.

    debug=True -> returns (new_corners, info) with per-side profiles, floor /
    peak / thresh / chosen-shift, the offset axis `ds`, the method, and the loose
    `ordered` corners — so visualisers plot EXACTLY what was computed here.
    """
    H, W = img_bgr.shape[:2]
    lab  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    gx = np.stack([cv2.Sobel(cv2.GaussianBlur(lab[:, :, c], (3, 3), 0),
                             cv2.CV_64F, 1, 0, ksize=3) for c in range(3)])  # (3,H,W)
    gy = np.stack([cv2.Sobel(cv2.GaussianBlur(lab[:, :, c], (3, 3), 0),
                             cv2.CV_64F, 0, 1, ksize=3) for c in range(3)])  # (3,H,W)

    ordered = order_corners(quad)
    cx, cy  = ordered[:, 0].mean(), ordered[:, 1].mean()
    diag    = np.linalg.norm(ordered[2] - ordered[0])
    d_in    = max(8, int(diag * inward_frac))
    d_out   = max(2, int(diag * outward_frac))
    ds      = np.arange(-d_out, d_in + 1)                   # ascending = outward->inward

    sides = []
    for i in range(4):
        p1, p2   = ordered[i], ordered[(i + 1) % 4]
        side_vec = p2 - p1
        side_len = np.linalg.norm(side_vec)
        s = dict(p1=p1, p2=p2, degenerate=bool(side_len < 1),
                 normal=np.zeros(2, np.float32), side_unit=np.zeros(2, np.float32),
                 base=None, dirg=None, valid=None, profile=None,
                 floor=np.nan, peak=np.nan, thresh=np.nan, shift=0.0)
        if s["degenerate"]:
            sides.append(s); continue

        side_unit = (side_vec / side_len).astype(np.float32)
        normal    = np.array([-side_unit[1], side_unit[0]], np.float32)
        mid       = (p1 + p2) / 2
        if np.dot(normal, np.array([cx, cy]) - mid) < 0:   # force INWARD
            normal = -normal
        s["side_unit"], s["normal"] = side_unit, normal

        ts   = np.linspace(0.08, 0.92, n_samples)
        base = p1[None, :] + ts[:, None] * side_vec[None, :]
        samp = base[None, :, :] + ds[:, None, None] * normal[None, None, :]
        sx = np.round(samp[..., 0]).astype(int)
        sy = np.round(samp[..., 1]).astype(int)
        valid = (sx >= 0) & (sx < W) & (sy >= 0) & (sy < H)
        sxx, syy = np.clip(sx, 0, W - 1), np.clip(sy, 0, H - 1)

        # directional gradient per channel, combined across L,a,b via L2 norm
        dir_ch = np.abs(gx[:, syy, sxx] * normal[0] + gy[:, syy, sxx] * normal[1])  # (3,D,n)
        dirg   = np.sqrt((dir_ch ** 2).sum(axis=0))                                  # (D,n)
        dirg[~valid] = 0
        profile = dirg.sum(axis=1) / np.maximum(valid.sum(axis=1), 1)
        if len(profile) >= 5:
            profile = np.convolve(profile, np.ones(5) / 5, mode="same")

        vmask = valid.sum(axis=1) > (n_samples * 0.5)
        floor = peak = thresh = np.nan
        shift = 0.0
        if vmask.sum() >= 3:
            pv    = profile[vmask]
            floor = float(np.percentile(pv, 15))            # background (table/fabric)
            peak  = float(pv.max())
            if peak > floor * 1.10 + 1e-6:                  # a real edge exists
                thresh = floor + peak_frac * (peak - floor)
                cross  = np.where((profile >= thresh) & vmask)[0]   # FIRST/outermost crossing
                shift  = float(ds[cross[0]]) if len(cross) else 0.0
        s.update(base=base, dirg=dirg, valid=valid, profile=profile,
                 floor=floor, peak=peak, thresh=thresh, shift=shift)
        sides.append(s)

    # ── Build a (point, direction) line for each side ─────────────────────────
    def _offset_line(s):
        """Single global offset: the loose side shifted perpendicular by shift."""
        mid = (s["p1"] + s["p2"]) / 2
        if s["degenerate"]:
            return mid.astype(np.float32), s["side_unit"]
        return (mid + s["shift"] * s["normal"]).astype(np.float32), s["side_unit"]

    def _linefit_line(s):
        """Per-along-side edge crossing + robust (Huber) line fit -> non-parallel ok."""
        if s["degenerate"] or np.isnan(s["thresh"]):
            return _offset_line(s)
        dirg, valid, base, normal = s["dirg"], s["valid"], s["base"], s["normal"]
        n = dirg.shape[1]
        offs = np.full(n, np.nan)
        for j in range(n):                                  # per along-side sample
            col = dirg[:, j].copy(); col[~valid[:, j]] = -1.0
            cidx = np.where(col >= s["thresh"])[0]          # first crossing of the robust thresh
            if len(cidx):
                offs[j] = ds[cidx[0]]
        good = ~np.isnan(offs)
        if good.sum() >= max(8, int(n * 0.25)):
            med = np.median(offs[good])
            mad = np.median(np.abs(offs[good] - med)) + 1e-6
            keep = good & (np.abs(offs - med) <= 3.0 * 1.4826 * mad)   # drop gross outliers
            if keep.sum() >= 8:
                pts = (base[keep] + offs[keep, None] * normal[None, :]).astype(np.float32)
                vx, vy, x0, y0 = cv2.fitLine(pts, cv2.DIST_HUBER, 0, 0.01, 0.01).ravel()
                return np.array([x0, y0], np.float32), np.array([vx, vy], np.float32)
        return _offset_line(s)                              # not enough inliers -> fall back

    def _select_aspect_shifts(target=CARD_ASPECT_TARGET, max_k=max_k,
                              cand_frac=cand_frac, tol=aspect_tol):
        """Choose, per side, the candidate edge offset whose 4-way combination
        (a) has quad aspect within `tol` of `target` (~1.4), AND (b) has a
        UNIFORM gradient along every chosen side. Uniformity = the MEDIAN
        along-side gradient at that offset (high only if MOST of the side sits on
        a real edge) — this rejects partial-length spikes (a stand, a shadow,
        interior text/art spanning only part of a side) that a MEAN cannot tell
        from a full-length border. Among feasible combos it prefers the OUTERMOST
        edges (the card border, not an inner divider). Overrides the offset-mode
        shifts only when a feasible combo is found."""
        tl, tr, br, bl = ordered
        base_w = (np.linalg.norm(tr - tl) + np.linalg.norm(br - bl)) / 2.0
        base_h = (np.linalg.norm(bl - tl) + np.linalg.norm(br - tr)) / 2.0

        def cands(s):
            base_off = float(s["shift"])
            if s["degenerate"] or s["profile"] is None or np.isnan(s["floor"]):
                return [(base_off, 1.0)]
            prof, floor, peak = s["profile"], s["floor"], s["peak"]
            dd = s["dirg"].astype(float); dd[~s["valid"]] = np.nan
            med = np.nan_to_num(np.nanmedian(dd, axis=1))      # along-side MEDIAN per offset
            rng = max(peak - floor, 1e-6)
            thr = floor + cand_frac * (peak - floor)
            peaks = [k for k in range(1, len(prof) - 1)
                     if prof[k] >= thr and prof[k] >= prof[k - 1] and prof[k] >= prof[k + 1]]
            peaks.sort(key=lambda k: -prof[k])                 # strongest mean peaks first
            offs, out = [], []
            def add(off):
                if all(abs(off - o) > 10 for o in offs):       # dedupe nearby offsets
                    di   = int(np.clip(np.searchsorted(ds, off), 0, len(ds) - 1))
                    cons = float(np.clip(med[di] / max(peak, 1e-6), 0.0, 1.0))   # uniformity
                    offs.append(off); out.append((off, cons))
            add(base_off)                                      # offset-mode pick always considered
            for k in peaks:
                add(float(ds[k]))
                if len(out) >= max_k + 1:
                    break
            return out

        cT, cR, cB, cL = cands(sides[0]), cands(sides[1]), cands(sides[2]), cands(sides[3])
        best, best_key, best_feasible = None, None, False
        for t in cT:
            for r in cR:
                for b in cB:
                    for l in cL:
                        W = base_w - l[0] - r[0]
                        H = base_h - t[0] - b[0]
                        if W <= 10 or H <= 10:
                            continue
                        aspect   = max(W, H) / min(W, H)
                        aerr     = abs(aspect - target) / target
                        feasible = aerr <= tol
                        cmin = min(t[1], r[1], b[1], l[1])     # weakest side must still be uniform
                        cavg = (t[1] + r[1] + b[1] + l[1]) / 4.0
                        total_off = t[0] + r[0] + b[0] + l[0]  # smaller => more OUTWARD
                        key = (1 if feasible else 0, round(cmin, 2),
                               -total_off, round(cavg, 3), -aerr)
                        if best_key is None or key > best_key:
                            best_key, best, best_feasible = key, (t[0], r[0], b[0], l[0]), feasible
        if best is not None and best_feasible:
            for s, sh in zip(sides, best):
                s["shift"] = float(sh)

    def _refine_uniform(window=refine_window):
        """Snap each side to the nearby offset that MAXIMISES the along-line
        gradient uniformity (median over the along-side samples). The first-rise
        pick sits on the OUTER flank of the edge (slightly loose); the median is
        maximal at the edge CENTRE = the true border, so this tightens it. A
        small `window` keeps it from hopping to a different edge."""
        for s in sides:
            if s["degenerate"] or s["dirg"] is None or s["valid"] is None:
                continue
            dd = s["dirg"].astype(float); dd[~s["valid"]] = np.nan
            med = np.nan_to_num(np.nanmedian(dd, axis=1))      # along-line uniformity per offset
            ci  = int(np.clip(np.searchsorted(ds, s["shift"]), 0, len(ds) - 1))
            lo, hi = max(0, ci - window), min(len(ds), ci + window + 1)
            seg = med[lo:hi]
            if seg.size and seg.max() > 0:
                s["shift"] = float(ds[lo + int(np.argmax(seg))])

    def _complete_with_aspect(target=CARD_ASPECT_TARGET, weak_ratio=0.5):
        """If ONE side has a weak/absent edge (low along-line uniformity at its
        chosen offset) while the other three are strong, DERIVE that side from the
        card aspect ratio instead of trusting its (invisible) gradient. Handles a
        low-contrast edge such as a dark card bottom on a dark background / stand.
        Needs 3 reliable sides; otherwise it leaves the shifts unchanged."""
        conf = []
        for s in sides:
            if s["degenerate"] or s["dirg"] is None or np.isnan(s["peak"]):
                conf.append(0.0); continue
            dd = s["dirg"].astype(float); dd[~s["valid"]] = np.nan
            med = np.nan_to_num(np.nanmedian(dd, axis=1))
            ci  = int(np.clip(np.searchsorted(ds, s["shift"]), 0, len(ds) - 1))
            conf.append(float(med[ci] / max(s["peak"], 1e-6)))
        conf = np.array(conf, dtype=float)
        weak = int(np.argmin(conf))
        others = np.delete(conf, weak)
        if others.size < 3 or np.median(others) <= 0:
            return
        if conf[weak] >= weak_ratio * np.median(others):
            return                              # no single clearly-weak side -> trust gradients
        tl, tr, br, bl = ordered
        base_w = (np.linalg.norm(tr - tl) + np.linalg.norm(br - bl)) / 2.0
        base_h = (np.linalg.norm(bl - tl) + np.linalg.norm(br - tr)) / 2.0
        sh = [s["shift"] for s in sides]        # [TOP, RIGHT, BOTTOM, LEFT]
        W  = base_w - sh[3] - sh[1]
        H  = base_h - sh[0] - sh[2]
        if W <= 1 or H <= 1:
            return
        if weak in (0, 2):                      # weak HORIZONTAL -> set height from width
            H_t    = W * target if H >= W else W / target
            strong = 2 if weak == 0 else 0      # keep the opposite (strong) horizontal side
            sides[weak]["shift"] = float((base_h - H_t) - sh[strong])
        else:                                   # weak VERTICAL -> set width from height
            W_t    = H / target if H >= W else H * target
            strong = 3 if weak == 1 else 1      # keep the opposite (strong) vertical side
            sides[weak]["shift"] = float((base_w - W_t) - sh[strong])

    if method == "aspect":
        _select_aspect_shifts()
        _refine_uniform()                  # precise snap onto the card border centre
        _complete_with_aspect()            # derive a low-contrast side from the ~1.4 ratio
        side_lines = [_offset_line(s) for s in sides]
    elif method == "linefit":
        side_lines = [_linefit_line(s) for s in sides]
    else:
        side_lines = [_offset_line(s) for s in sides]

    def line_intersect(a1, d1, a2, d2):
        cross = d1[0] * d2[1] - d1[1] * d2[0]
        if abs(cross) < 1e-8:
            return (a1 + a2) / 2
        diff = a2 - a1
        t = (diff[0] * d2[1] - diff[1] * d2[0]) / cross
        return (a1 + t * d1).astype(np.float32)

    new_corners = np.zeros((4, 2), np.float32)
    for i in range(4):
        prev = (i - 1) % 4
        new_corners[i] = line_intersect(side_lines[prev][0], side_lines[prev][1],
                                        side_lines[i][0], side_lines[i][1])
    new_corners[:, 0] = new_corners[:, 0].clip(0, W - 1)
    new_corners[:, 1] = new_corners[:, 1].clip(0, H - 1)

    if debug:
        info = dict(ordered=ordered, ds=ds, method=method,
                    names=["TOP", "RIGHT", "BOTTOM", "LEFT"],
                    sides=[dict(profile=s["profile"], floor=s["floor"], peak=s["peak"],
                                thresh=s["thresh"], shift=s["shift"]) for s in sides])
        return new_corners, info
    return new_corners


# ── Pokemon card aspect ratio (used for soft scoring only) ────────────────────
CARD_ASPECT_TARGET = 1.397   # 63×88mm → 88/63


def _hough_quad(edge_map: np.ndarray, h: int, w: int) -> Optional[np.ndarray]:
    """
    Fallback for busy backgrounds where MORPH_CLOSE floods the frame into one
    blob (coverage > 90% → rejected).  Finds the 4 dominant card-edge lines
    directly from the raw fused edge map and intersects them for corners —
    no closed contour required.  The loose result is refined by
    _tighten_quad_to_edges() inside the strategy.
    """
    min_side = min(h, w)
    lines = cv2.HoughLinesP(edge_map, rho=1, theta=np.pi / 180,
                            threshold=max(50, min_side // 10),
                            minLineLength=min_side * 0.20,
                            maxLineGap=min_side * 0.06)
    if lines is None or len(lines) < 4:
        return None

    h_lines, v_lines = [], []
    for x1, y1, x2, y2 in lines[:, 0]:
        angle = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
        if angle < 25 or angle > 155:      # near-horizontal
            h_lines.append((x1, y1, x2, y2))
        elif 65 < angle < 115:             # near-vertical
            v_lines.append((x1, y1, x2, y2))
    if len(h_lines) < 2 or len(v_lines) < 2:
        return None

    top_line    = min(h_lines, key=lambda l: (l[1] + l[3]) / 2)
    bottom_line = max(h_lines, key=lambda l: (l[1] + l[3]) / 2)
    left_line   = min(v_lines, key=lambda l: (l[0] + l[2]) / 2)
    right_line  = max(v_lines, key=lambda l: (l[0] + l[2]) / 2)

    def intersect(l1, l2):
        x1, y1, x2, y2 = l1; x3, y3, x4, y4 = l2
        p, d1 = np.array([x1, y1], np.float32), np.array([x2 - x1, y2 - y1], np.float32)
        q, d2 = np.array([x3, y3], np.float32), np.array([x4 - x3, y4 - y3], np.float32)
        cross = d1[0] * d2[1] - d1[1] * d2[0]
        if abs(cross) < 1e-8: return None
        t = ((q - p)[0] * d2[1] - (q - p)[1] * d2[0]) / cross
        return p + t * d1

    corners = [intersect(top_line, left_line),  intersect(top_line, right_line),
               intersect(bottom_line, right_line), intersect(bottom_line, left_line)]
    if any(c is None for c in corners):
        return None

    box = np.array(corners, dtype=np.float32)
    rect = cv2.minAreaRect(box)
    rw, rh = rect[1]
    if rw == 0 or rh == 0:
        return None
    aspect = max(rw, rh) / min(rw, rh)
    return box if 0.8 <= aspect <= 2.5 else None


def _multifilter_polygon_strategy(img_bgr: np.ndarray) -> Optional[np.ndarray]:
    """
    Strategy 1 — Multi-filter polygon detection (1vcian approach).

    Four complementary edge maps are fused:
      ① Adaptive threshold  — local contrast at any background luminance
      ② LAB L-channel Canny — lighting-invariant; ignores shadows
      ③ HSV saturation Canny— highlights colour-saturated card artwork edge
      ④ Bilateral Canny gray — smooth general-purpose detector

    Pipeline (matches the validated debug flow):
      • Adaptive MORPH_CLOSE kernel (3% of shorter dim) scales with resolution.
      • PROGRESSIVE close (3→6→10→16 iters): pick the FIRST pass that yields a
        card-like contour. Handles both clean and fragmented boundaries.
      • Hough-line FALLBACK: on busy backgrounds the close floods to a full-frame
        blob (rejected by coverage); _hough_quad() recovers a loose quad.

    Returns a LOOSE (4, 2) quad, or None. Tightening happens once, centrally, in
    loose_quad()'s consumers (opencv_find_card / visualize_tighten) — never here.
    """
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # ── ① Adaptive threshold ─────────────────────────────────────────────────
    blur_a  = cv2.GaussianBlur(gray, (7, 7), 0)
    adapt   = cv2.adaptiveThreshold(blur_a, 255,
                                     cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 11, 2)
    edges_a = cv2.Canny(adapt, 30, 100)

    # ── ② LAB L-channel (lighting-invariant) ────────────────────────────────
    l_blur  = cv2.bilateralFilter(
                  cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)[:, :, 0], 9, 75, 75)
    edges_l = cv2.Canny(l_blur, 20, 60)

    # ── ③ HSV saturation (colour-discriminant) ───────────────────────────────
    _, sat_t = cv2.threshold(
                   cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)[:, :, 1],
                   30, 255, cv2.THRESH_BINARY)
    edges_s  = cv2.Canny(sat_t, 30, 100)

    # ── ④ Bilateral Canny gray (general) ────────────────────────────────────
    bilateral = cv2.bilateralFilter(gray, 9, 75, 75)
    edges_c   = cv2.bitwise_or(cv2.Canny(bilateral, 30, 80),
                                cv2.Canny(bilateral, 10, 40))

    # ── Fuse + dilate ─────────────────────────────────────────────────────────
    combined_raw = edges_a | edges_l | edges_s | edges_c
    dilated = cv2.dilate(combined_raw,
                         cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)),
                         iterations=2)

    # ── Adaptive kernel scaled to resolution (~3% of shorter dimension) ───────
    close_k = max(15, int(min(h, w) * 0.03))

    def _score_contours(comb):
        contours, _ = cv2.findContours(comb, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours    = sorted(contours, key=cv2.contourArea, reverse=True)[:20]
        best_box, best_score = None, -1.0
        for cnt in contours:
            if cv2.contourArea(cnt) < h * w * 0.03:
                continue
            hull = cv2.convexHull(cnt)
            rect = cv2.minAreaRect(hull)
            rw, rh = rect[1]
            if rw == 0 or rh == 0:
                continue
            box      = cv2.boxPoints(rect).astype(np.float32)
            aspect   = max(rw, rh) / min(rw, rh)
            coverage = cv2.contourArea(box) / (h * w)
            if not (0.03 <= coverage <= 0.90):
                continue
            aspect_err = abs(aspect - CARD_ASPECT_TARGET) / CARD_ASPECT_TARGET
            score = (0.60 * min(coverage / 0.70, 1.0) +
                     0.40 * max(0.0, 1.0 - aspect_err / 0.50))
            if score > best_score:
                best_score, best_box = score, box
        return best_box, best_score

    # ── Progressive MORPH_CLOSE: first pass with a card contour wins ───────────
    best_corners, best_score, found_at = None, -1.0, None
    for close_iters in (3, 6, 10, 16):
        comb = cv2.morphologyEx(
            dilated, cv2.MORPH_CLOSE,
            cv2.getStructuringElement(cv2.MORPH_RECT, (close_k, close_k)),
            iterations=close_iters)
        box, score = _score_contours(comb)
        if box is not None:
            best_corners, best_score, found_at = box, score, f"close×{close_iters}"
            break

    # ── Hough-line fallback (busy backgrounds) ────────────────────────────────
    if best_corners is None:
        best_corners = _hough_quad(combined_raw, h, w)
        if best_corners is not None:
            found_at = "hough"

    if best_corners is None:
        if VERBOSE:
            print(f"  ↳ multifilter: no quad (close_k={close_k}px, hough also failed)")
        return None

    if VERBOSE:
        cov = cv2.contourArea(best_corners) / (h * w) * 100
        print(f"  ↳ multifilter[{found_at}]: loose quad coverage={cov:.1f}%")

    return best_corners   # LOOSE quad — tightened centrally by loose_quad's consumers


def _grabcut_strategy(img_bgr: np.ndarray) -> Optional[np.ndarray]:
    """
    Strategy 2 — GrabCut foreground segmentation.
    Works for full-art / borderless cards where edge detection finds no
    clear perimeter (e.g. Charizard ex on any background).
    GrabCut rect initialised with 10% inset from each edge.
    """
    h, w   = img_bgr.shape[:2]
    pad    = 0.10
    rect   = (int(w * pad), int(h * pad),
              int(w * (1 - 2*pad)), int(h * (1 - 2*pad)))
    mask      = np.zeros((h, w), np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(img_bgr, mask, rect, bgd_model, fgd_model,
                    5, cv2.GC_INIT_WITH_RECT)
    except cv2.error:
        return None
    fg_mask = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD),
                       255, 0).astype(np.uint8)
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN,  kernel, iterations=1)
    return _quad_from_mask(fg_mask)


def _corner_color_strategy(img_bgr: np.ndarray) -> Optional[np.ndarray]:
    """
    Strategy 3 — Border-strip background colour subtraction.
    Samples thin strips along all 4 edges as background model,
    thresholds foreground pixels, fits minAreaRect to largest contour.
    """
    h, w  = img_bgr.shape[:2]
    lab   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    strip = max(12, int(min(h, w) * 0.05))
    samples = np.vstack([
        lab[:strip,  :  ].reshape(-1, 3),
        lab[-strip:, :  ].reshape(-1, 3),
        lab[:,  :strip  ].reshape(-1, 3),
        lab[:, -strip:  ].reshape(-1, 3),
    ])
    bg_mean = np.median(samples, axis=0)
    bg_std  = np.abs(samples - bg_mean).mean(axis=0).clip(min=6, max=25)
    diff    = np.abs(lab - bg_mean)
    fg_mask = ((diff / bg_std).max(axis=2) >= 2.0).astype(np.uint8) * 255
    close_k = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    open_k  = cv2.getStructuringElement(cv2.MORPH_RECT, (9,  9))
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, close_k, iterations=4)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN,  open_k,  iterations=1)
    fg_mask[:strip,  :] = 0; fg_mask[-strip:, :] = 0
    fg_mask[:,  :strip] = 0; fg_mask[:, -strip:] = 0
    return _quad_from_mask(fg_mask)


def opencv_find_card(img_bgr: np.ndarray) -> Optional[np.ndarray]:
    """
    Multi-strategy card boundary detector — no trained model required.

    ONE shared path (identical to what visualize_tighten plots):
      loose_quad()             -> cascade multifilter -> grabcut -> corner_colour;
                                  first card-like LOOSE quad wins
      _tighten_quad_to_edges() -> snap each side to the true outer border

    Returns (4, 2) float32 corners [TL, TR, BR, BL], or None.
    """
    h, w = img_bgr.shape[:2]
    raw, src = loose_quad(img_bgr)
    if raw is None:
        if VERBOSE: print("  All strategies failed")
        return None
    corners = _tighten_quad_to_edges(img_bgr, raw)
    if VERBOSE:
        before = cv2.contourArea(raw.astype(np.float32)) / (h * w) * 100
        after  = cv2.contourArea(corners.astype(np.float32)) / (h * w) * 100
        print(f"  {src}: accepted (area {before:.1f}% -> {after:.1f}% after tighten)")
    return corners


def detect_and_warp(image_path: str | Path, conf: float = CONF_THRESHOLD) -> dict:
    """
    Detect card boundary and perspective-warp to CARD_W × CARD_H.
    Priority: YOLO OBB (trained model) → opencv_find_card() fallback.
    Returns dict: original, crops, corners, scores, methods, n_detected.
    """
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")
    img_h, img_w = img_bgr.shape[:2]
    crops, corners_list, scores, methods = [], [], [], []

    yolo_accepted = 0
    try:
        yolo_res = model.predict(img_bgr, conf=conf, verbose=False)
        yolo_obb = yolo_res[0].obb
        if yolo_obb is not None:
            for i in range(len(yolo_obb)):
                corners_px = yolo_obb.xyxyxyxy[i].cpu().numpy()
                score      = float(yolo_obb.conf[i].cpu())
                if not _is_card_like(corners_px, img_h, img_w):
                    if VERBOSE:
                        a = cv2.contourArea(corners_px.astype(np.float32))/(img_h*img_w)*100
                        print(f"  ⚠️  YOLO det #{i} rejected (area={a:.1f}%)")
                    continue
                crops.append(perspective_warp(img_bgr, corners_px))
                corners_list.append(corners_px); scores.append(score)
                methods.append("yolo");          yolo_accepted += 1
    except Exception as e:
        if VERBOSE: print(f"  ⚠️  YOLO error: {e}")

    if yolo_accepted == 0:
        if VERBOSE: print("  🔄 No valid YOLO detections → OpenCV fallback")
        cv_corners = opencv_find_card(img_bgr)
        if cv_corners is not None:
            crops.append(perspective_warp(img_bgr, cv_corners))
            corners_list.append(cv_corners); scores.append(1.0); methods.append("opencv")

    return dict(original=img_bgr, crops=crops, corners=corners_list,
                scores=scores, methods=methods, n_detected=len(crops))


def loose_quad(img_bgr: np.ndarray):
    """
    Return the LOOSE (pre-tighten) quad from the first cascade strategy that
    yields a card-like shape, plus its source name. This is THE cascade — both
    opencv_find_card() and visualize_tighten() call it, so they stay in sync.
    """
    h, w = img_bgr.shape[:2]
    q = _multifilter_polygon_strategy(img_bgr)
    if q is not None and _is_card_like(q, h, w):
        return q, "multifilter"
    for name, fn in (("grabcut", _grabcut_strategy), ("corner_colour", _corner_color_strategy)):
        q = fn(img_bgr)
        if q is not None and _is_card_like(q, h, w):
            return q, name
    return None, None


def visualize_tighten(image_path, peak_frac: Optional[float] = None, method: str = "offset",
                      aspect_tol: Optional[float] = None, cand_frac: Optional[float] = None,
                      max_k: Optional[int] = None, refine_window: Optional[int] = None):
    """
    Single-image tighten diagnostic. Shows the loose (red) vs tightened (green)
    quad, the resulting warp, and the four per-side gradient profiles WITH the
    exact floor / threshold / chosen-offset that _tighten_quad_to_edges() used
    (debug=True) — so the plot can never drift from the live function.

    peak_frac=None uses the function's default (single source of truth); pass a
    value only to experiment.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        print(f"cannot read {image_path}"); return None
    raw, src = loose_quad(img)
    if raw is None:
        print("no strategy produced a loose quad - detection failed, not tighten."); return None

    kw = {k: v for k, v in dict(peak_frac=peak_frac, aspect_tol=aspect_tol,
                               cand_frac=cand_frac, max_k=max_k,
                               refine_window=refine_window).items() if v is not None}
    tgt, info = _tighten_quad_to_edges(img, raw, debug=True, method=method, **kw)
    overrides = ", ".join(f"{k}={v}" for k, v in kw.items()) or "all defaults"
    print(f"loose quad from: {src}   method={method}   ({overrides})")

    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    vis = img.copy()
    cv2.polylines(vis, [order_corners(raw).astype(np.int32)], True, (0, 0, 255), 2)   # red  = loose
    cv2.polylines(vis, [order_corners(tgt).astype(np.int32)], True, (0, 255, 0), 3)   # green= tightened
    axes[0, 0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title("red = loose   green = tightened"); axes[0, 0].axis("off")
    axes[1, 0].imshow(cv2.cvtColor(perspective_warp(img, tgt), cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title("warp (tightened)"); axes[1, 0].axis("off")

    ds = info["ds"]
    for ax, name, s in zip([axes[0, 1], axes[0, 2], axes[1, 1], axes[1, 2]],
                           info["names"], info["sides"]):
        if s["profile"] is None:
            ax.set_title(f"{name}: degenerate side"); ax.axis("off"); continue
        ax.plot(ds, s["profile"])
        if not np.isnan(s["thresh"]):
            ax.axhline(s["thresh"], color="r", ls="--", label=f"thr={s['thresh']:.0f}")
        ax.axvline(s["shift"], color="g", label=f"chosen d={s['shift']:.0f}")
        ax.axvline(0, color="gray", ls=":")
        ax.set_title(f"{name}  floor={s['floor']:.0f} peak={s['peak']:.0f}")
        ax.legend(fontsize=7)
    plt.suptitle(f"{Path(image_path).name}  (tighten diagnostic)", fontsize=11)
    plt.tight_layout(); plt.show()
    return dict(raw=raw, tightened=tgt, source=src, info=info)


def find_inner_frame(img_bgr: np.ndarray, outer_quad: np.ndarray,
                     max_border_frac: float = 0.14, min_border_px: int = 3,
                     n_samples: int = 60, peak_frac: float = 0.40):
    """
    Given the OUTER card quad, find the INNER printed frame. Every Pokemon card
    has a border, so just inside the physical edge there is a second strong,
    UNIFORM, parallel edge. For each side we search INWARD from the outer edge
    for that edge; the gap is the border width.

    Returns dict:
      inner_quad   — (4,2) inner-frame corners
      widths       — per-side border width px [TOP, RIGHT, BOTTOM, LEFT] (NaN if none)
      found        — per-side bool
      perim_ratio  — perimeter(outer) / perimeter(inner)   (~1.10-1.20 for a card)
      width_cv     — coeff. of variation of the 4 widths (low = consistent border)
    """
    H, W = img_bgr.shape[:2]
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    gx = np.stack([cv2.Sobel(cv2.GaussianBlur(lab[:, :, c], (3, 3), 0),
                             cv2.CV_64F, 1, 0, ksize=3) for c in range(3)])
    gy = np.stack([cv2.Sobel(cv2.GaussianBlur(lab[:, :, c], (3, 3), 0),
                             cv2.CV_64F, 0, 1, ksize=3) for c in range(3)])

    ordered = order_corners(outer_quad)
    cx, cy  = ordered[:, 0].mean(), ordered[:, 1].mean()
    short   = min(np.linalg.norm(ordered[1] - ordered[0]),
                  np.linalg.norm(ordered[3] - ordered[0]))
    dmax    = max(min_border_px + 6, int(short * max_border_frac))
    ds      = np.arange(0, dmax + 1)

    widths, normals, found = [], [], []
    for i in range(4):
        p1, p2 = ordered[i], ordered[(i + 1) % 4]
        sv = p2 - p1; L = np.linalg.norm(sv)
        if L < 1:
            widths.append(np.nan); normals.append(np.zeros(2, np.float32)); found.append(False); continue
        su = (sv / L).astype(np.float32)
        normal = np.array([-su[1], su[0]], np.float32)
        mid = (p1 + p2) / 2
        if np.dot(normal, np.array([cx, cy]) - mid) < 0:   # inward
            normal = -normal
        normals.append(normal)

        ts = np.linspace(0.10, 0.90, n_samples)
        base = p1[None, :] + ts[:, None] * sv[None, :]
        samp = base[None, :, :] + ds[:, None, None] * normal[None, None, :]
        sx = np.round(samp[..., 0]).astype(int); sy = np.round(samp[..., 1]).astype(int)
        valid = (sx >= 0) & (sx < W) & (sy >= 0) & (sy < H)
        sxx, syy = np.clip(sx, 0, W - 1), np.clip(sy, 0, H - 1)
        dir_ch = np.abs(gx[:, syy, sxx] * normal[0] + gy[:, syy, sxx] * normal[1])
        dirg = np.sqrt((dir_ch ** 2).sum(axis=0))
        dd = dirg.astype(float); dd[~valid] = np.nan
        med = np.nan_to_num(np.nanmedian(dd, axis=1))      # along-line uniformity inward
        mx = float(med.max())
        if mx <= 1e-6:
            widths.append(np.nan); found.append(False); continue
        thr = peak_frac * mx
        # A real inner frame is a uniform peak that comes AFTER a low-gradient
        # margin (the printed border). Require the gradient to first DROP (the
        # margin) before accepting a peak — this rejects the outer-edge shoulder
        # that otherwise gives a bogus width ~= min_border_px on every side.
        seen_gap, wpx = False, None
        for k in range(min_border_px, len(med) - 1):
            if med[k] < 0.5 * thr:
                seen_gap = True
            elif seen_gap and med[k] >= thr and med[k] >= med[k - 1] and med[k] >= med[k + 1]:
                wpx = float(ds[k]); break
        if wpx is not None:
            widths.append(wpx); found.append(True)
        else:
            widths.append(np.nan); found.append(False)

    found_w = [w for w, f in zip(widths, found) if f and not np.isnan(w)]
    fill = float(np.mean(found_w)) if found_w else 0.0
    use_w = [w if (f and not np.isnan(w)) else fill for w, f in zip(widths, found)]

    def _line(i, wpx):
        mid = (ordered[i] + ordered[(i + 1) % 4]) / 2 + wpx * normals[i]
        su = ordered[(i + 1) % 4] - ordered[i]; n = np.linalg.norm(su)
        return mid.astype(np.float32), (su / n if n > 0 else su).astype(np.float32)

    def _ix(a1, d1, a2, d2):
        cr = d1[0] * d2[1] - d1[1] * d2[0]
        if abs(cr) < 1e-8: return (a1 + a2) / 2
        diff = a2 - a1; t = (diff[0] * d2[1] - diff[1] * d2[0]) / cr
        return (a1 + t * d1).astype(np.float32)

    lines = [_line(i, use_w[i]) for i in range(4)]
    inner = np.zeros((4, 2), np.float32)
    for i in range(4):
        prev = (i - 1) % 4
        inner[i] = _ix(lines[prev][0], lines[prev][1], lines[i][0], lines[i][1])

    def _perim(q):
        q = order_corners(q)
        return float(sum(np.linalg.norm(q[(j + 1) % 4] - q[j]) for j in range(4)))
    po, pi = _perim(ordered), _perim(inner)
    perim_ratio = po / pi if pi > 0 else np.nan
    width_cv = float(np.std(found_w) / np.mean(found_w)) if len(found_w) >= 2 and np.mean(found_w) > 0 else np.nan
    return dict(inner_quad=inner, widths=widths, found=found,
                perim_ratio=perim_ratio, width_cv=width_cv)


def card_border_score(img_bgr: np.ndarray, outer_quad: np.ndarray,
                      ratio_lo: float = 1.05, ratio_hi: float = 1.30,
                      cv_max: float = 0.40, aspect_tol: float = 0.12) -> dict:
    """
    Card discriminant combining BOTH cues the user identified:
      1. outer aspect ratio close to ~1.4 (CARD_ASPECT_TARGET)
      2. a printed border: outer/inner PERIMETER ratio within [ratio_lo, ratio_hi]
         and the 4 border widths reasonably consistent (width_cv <= cv_max).
    Returns find_inner_frame(...) plus aspect, aspect_err, is_card.
    """
    info = find_inner_frame(img_bgr, outer_quad)
    oc = order_corners(outer_quad)
    w = (np.linalg.norm(oc[1] - oc[0]) + np.linalg.norm(oc[2] - oc[3])) / 2.0
    h = (np.linalg.norm(oc[3] - oc[0]) + np.linalg.norm(oc[2] - oc[1])) / 2.0
    aspect = max(w, h) / min(w, h) if min(w, h) > 0 else 0.0
    aerr   = abs(aspect - CARD_ASPECT_TARGET) / CARD_ASPECT_TARGET
    pr, cv = info["perim_ratio"], info["width_cv"]
    is_card = (aerr <= aspect_tol and
               not np.isnan(pr) and ratio_lo <= pr <= ratio_hi and
               (np.isnan(cv) or cv <= cv_max))
    info.update(aspect=aspect, aspect_err=aerr, is_card=bool(is_card))
    return info


def visualize_card_border(image_path, method: str = "aspect", **tighten_kw):
    """
    Second-stage diagnostic: detect -> tighten OUTER quad -> find INNER frame.
    Draws outer (green) + inner (cyan) boxes and prints aspect, per-side border
    widths, outer/inner perimeter ratio, and the is_card verdict — use it to
    calibrate the ratio / cv thresholds before wiring this in as a hard gate.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        print(f"cannot read {image_path}"); return None
    raw, srcname = loose_quad(img)
    if raw is None:
        print("no loose quad - detection failed."); return None
    outer = _tighten_quad_to_edges(img, raw, method=method, **tighten_kw)
    info  = card_border_score(img, outer)

    vis = img.copy()
    cv2.polylines(vis, [order_corners(outer).astype(np.int32)], True, (0, 255, 0), 3)        # green outer
    cv2.polylines(vis, [order_corners(info["inner_quad"]).astype(np.int32)], True, (255, 255, 0), 2)  # cyan inner
    fig, ax = plt.subplots(1, 2, figsize=(13, 8))
    ax[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax[0].set_title(f"green=outer  cyan=inner frame   is_card={info['is_card']}"); ax[0].axis("off")
    ax[1].imshow(cv2.cvtColor(perspective_warp(img, outer), cv2.COLOR_BGR2RGB))
    ax[1].set_title("warp (outer)"); ax[1].axis("off")
    plt.suptitle(f"{Path(image_path).name}  (border check, method={method})", fontsize=11)
    plt.tight_layout(); plt.show()

    bw = [None if np.isnan(w) else round(w, 1) for w in info["widths"]]
    print(f"source={srcname}  method={method}")
    print(f"  aspect      = {info['aspect']:.3f}   (target {CARD_ASPECT_TARGET:.3f}, err {info['aspect_err']*100:.1f}%)")
    print(f"  border px   = {bw}   [TOP, RIGHT, BOTTOM, LEFT]")
    print(f"  width_cv    = " + ("n/a" if np.isnan(info['width_cv']) else f"{info['width_cv']:.3f}"))
    print(f"  perim ratio = " + ("n/a" if np.isnan(info['perim_ratio']) else f"{info['perim_ratio']:.3f}") + "   (outer / inner)")
    print(f"  => is_card  = {info['is_card']}")
    return info


print("Detection helpers loaded — cascade: multifilter_polygon (progressive close + Hough + "
      "self-tighten) → grabcut → corner_colour.")

In [ ]:
# ── Quick tighten diagnostic ──────────────────────────────────────────────────
# visualize_tighten() reuses the live _tighten_quad_to_edges settings (debug=True),
# so this plot can never drift from what opencv_find_card produces.
# Pass peak_frac=... ONLY to experiment; leave it out to use the pipeline default.
DEBUG_IMAGE = "ungraded/ex_0_front.jpeg"

result = visualize_tighten(DEBUG_IMAGE)   # add e.g. peak_frac=0.4 to test a value

In [ ]:
# ── Batch validation: loose vs tightened across many images ───────────────────
# Uses the SAME settings as the live pipeline (loose_quad + default tighten) —
# no per-cell overrides, so nothing here can disagree with opencv_find_card.
import glob

TEST_GLOBS = ["cards/*.jpeg", "ungraded/*.jpeg"]   # adjust to your folders
test_paths = sorted({p for g in TEST_GLOBS for p in glob.glob(g)})
print(f"Validating {len(test_paths)} images (peak_frac = pipeline default)\n")

n = max(len(test_paths), 1)
fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
if n == 1:
    axes = axes.reshape(1, 3)

rows = []
for r, path in enumerate(test_paths):
    img = cv2.imread(path)
    if img is None:
        for c in range(3): axes[r, c].axis("off")
        axes[r, 0].set_title(f"{Path(path).name}\n(unreadable)", color="red")
        continue
    H, W = img.shape[:2]
    raw, src = loose_quad(img)
    tgt = _tighten_quad_to_edges(img, raw) if raw is not None else None   # default peak_frac

    vis = img.copy()
    if raw is not None: cv2.polylines(vis, [order_corners(raw).astype(np.int32)], True, (0, 0, 255), 2)
    if tgt is not None: cv2.polylines(vis, [order_corners(tgt).astype(np.int32)], True, (0, 255, 0), 3)
    axes[r, 0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[r, 0].set_title(f"{Path(path).name}  [{src}]\nred=loose  green=tightened", fontsize=9)
    axes[r, 0].axis("off")

    if raw is not None: axes[r, 1].imshow(cv2.cvtColor(perspective_warp(img, raw), cv2.COLOR_BGR2RGB))
    axes[r, 1].set_title("warp - loose", fontsize=9); axes[r, 1].axis("off")
    if tgt is not None: axes[r, 2].imshow(cv2.cvtColor(perspective_warp(img, tgt), cv2.COLOR_BGR2RGB))
    axes[r, 2].set_title("warp - tightened", fontsize=9); axes[r, 2].axis("off")

    cov_r = cv2.contourArea(raw) / (H * W) * 100 if raw is not None else 0
    cov_t = cv2.contourArea(tgt) / (H * W) * 100 if tgt is not None else 0
    ok    = _is_card_like(tgt, H, W) if tgt is not None else False
    rows.append((Path(path).name, str(src), cov_r, cov_t, ok))

plt.tight_layout(); plt.show()

print(f"{'image':<26}{'source':<14}{'loose%':>8}{'tight%':>8}  card_like")
print("-" * 64)
for name, src, cr, ct, ok in rows:
    print(f"{name:<26}{src:<14}{cr:8.1f}{ct:8.1f}  {'OK' if ok else 'XX'}")

### A3 — Visualise detections on test images

In [ ]:
def draw_obb(img_bgr: np.ndarray, corners_list: list, scores: list,
             methods: list) -> np.ndarray:
    """Draw OBB polygons + confidence scores + method label on a copy of the image."""
    vis = img_bgr.copy()
    colors = {"yolo": (0, 255, 255), "opencv": (0, 200, 255)}
    for corners, score, method in zip(corners_list, scores, methods):
        color = colors.get(method, (255, 255, 0))
        pts   = order_corners(corners).astype(np.int32)
        cv2.polylines(vis, [pts], isClosed=True, color=color, thickness=3)
        label = f"{method} {score:.2f}" if method == "yolo" else method
        cv2.putText(vis, label, tuple(pts[0]),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
    return vis


def visualise_detection(image_path: str | Path, conf: float = CONF_THRESHOLD):
    """Detect (YOLO or OpenCV fallback), draw overlay, show original + warped crops."""
    det = detect_and_warp(image_path, conf)
    n   = det["n_detected"]

    print(f"{'─'*60}")
    print(f"Image: {Path(image_path).name}  |  Detected: {n}  "
          f"|  Methods: {det.get('methods', [])}")

    n_cols = 1 + max(n, 1)
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 7))

    # Original with overlay
    vis = draw_obb(det["original"], det["corners"], det["scores"],
                   det.get("methods", ["?"] * n))
    axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Detection ({n} found)", fontsize=10)
    axes[0].axis("off")

    if n == 0:
        axes[1].text(0.5, 0.5, "No card found\nTry a cleaner background",
                     ha="center", va="center", fontsize=11)
        axes[1].axis("off")
    else:
        for i, (crop, method, score) in enumerate(
                zip(det["crops"], det["methods"], det["scores"])):
            axes[i + 1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            conf_str = f"conf={score:.2f}" if method == "yolo" else "OpenCV"
            axes[i + 1].set_title(f"Crop {i}  [{method}]  {conf_str}", fontsize=10)
            axes[i + 1].axis("off")

    for j in range(n + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
    return det


# Run on all test images
all_detections = {}
for img_path in TEST_IMAGES:
    if Path(img_path).exists():
        all_detections[img_path] = visualise_detection(img_path)
    else:
        print(f"⚠️  Not found: {img_path}")

### A4 — Save warped crops to disk

In [ ]:
OUT_CROPS_DIR = Path("crops_obb")
OUT_CROPS_DIR.mkdir(exist_ok=True)

saved = []
for img_path, det in all_detections.items():
    stem = Path(img_path).stem
    for i, crop in enumerate(det["crops"]):
        out_path = OUT_CROPS_DIR / f"{stem}_card{i}.jpg"
        cv2.imwrite(str(out_path), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
        saved.append(out_path)
        print(f"  Saved: {out_path}")

print(f"\nTotal crops saved: {len(saved)}")

### A5 — Feed crops into our existing centering detector

Demonstrates the full pipeline: OBB extract → warp → centering analysis.

In [ ]:
try:
    import imagehash
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "imagehash", "-q"])
    import imagehash

# ── Config ────────────────────────────────────────────────────────────────────
PHASH_HASH_SIZE   = 8      # 8×8 DCT → 64-bit hash  (increase to 16 for 256-bit)
PHASH_MATCH_DIST  = 12     # Hamming distance ≤ 12 counts as a match (out of 64)
PHASH_CACHE_FILE  = Path("datasets/phash_db.pkl")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}


# ── Build / load the pHash database ──────────────────────────────────────────
def build_phash_db(cards_dir: Path = Path("datasets"),
                   hash_size: int   = PHASH_HASH_SIZE,
                   cache_file: Path = PHASH_CACHE_FILE,
                   force_rebuild: bool = False) -> dict:
    """
    Compute pHash for every card image under `cards_dir` and return a dict:
      { Path: imagehash.ImageHash }

    Results are cached to `cache_file` (pickle) so subsequent calls are instant.
    Set force_rebuild=True to regenerate from scratch.
    """
    import pickle

    if cache_file.exists() and not force_rebuild:
        with open(cache_file, "rb") as f:
            db = pickle.load(f)
        print(f"✅  Loaded pHash DB: {len(db)} cards from {cache_file}")
        return db

    card_paths = [
        p for p in cards_dir.rglob("*")
        if p.suffix.lower() in IMG_EXTS
        and "yolo_obb_cards" not in str(p)
        and "backgrounds"    not in str(p)
    ]

    db = {}
    errors = 0
    for p in tqdm(card_paths, desc="Computing pHashes"):
        try:
            img  = Image.open(p).convert("RGB")
            db[p] = imagehash.phash(img, hash_size=hash_size)
        except Exception:
            errors += 1

    cache_file.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_file, "wb") as f:
        pickle.dump(db, f)

    print(f"✅  Built pHash DB: {len(db)} cards ({errors} errors) → {cache_file}")
    return db


phash_db = build_phash_db()


# ── Query function ────────────────────────────────────────────────────────────
def identify_card(crop_bgr: np.ndarray,
                  db: dict        = None,
                  max_dist: int   = PHASH_MATCH_DIST,
                  hash_size: int  = PHASH_HASH_SIZE,
                  top_k: int      = 3) -> list[dict]:
    """
    Identify a warped card crop by comparing its pHash against the database.

    Returns a list of up to top_k matches, each dict:
      { path, distance, confidence }

    confidence:
      "high"   — distance ≤ 6   (≤9% bits differ)
      "medium" — distance ≤ 12  (≤19% bits differ)
      "low"    — distance ≤ max_dist
      None     — no match found within max_dist
    """
    if db is None:
        db = phash_db

    # Convert BGR crop → PIL RGB for hashing
    crop_pil  = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    query_hash = imagehash.phash(crop_pil, hash_size=hash_size)

    # Compute Hamming distance to every DB entry
    results = []
    for path, ref_hash in db.items():
        dist = query_hash - ref_hash   # imagehash overloads − as Hamming distance
        if dist <= max_dist:
            results.append({"path": path, "distance": dist})

    # Sort by distance (ascending) and return top_k
    results.sort(key=lambda x: x["distance"])
    results = results[:top_k]

    for r in results:
        d = r["distance"]
        r["confidence"] = "high" if d <= 6 else "medium" if d <= 12 else "low"

    return results


# ── Run identification on all saved crops ────────────────────────────────────
print("\nCard Identification via pHash")
print("═" * 60)
for crop_path in saved:
    crop_bgr = cv2.imread(str(crop_path))
    if crop_bgr is None:
        continue

    matches = identify_card(crop_bgr)

    print(f"\n📷  {crop_path.name}")
    if not matches:
        print("  ❌ No match found (try raising PHASH_MATCH_DIST)")
    else:
        for i, m in enumerate(matches):
            icon = "🥇" if i == 0 else f"  #{i+1}"
            print(f"  {icon}  {m['path'].name:<40}  "
                  f"dist={m['distance']:2d}/64  [{m['confidence']}]")

### A6 — Card Identification via Perceptual Hash

Once a card is extracted and warped, we can **identify which card it is** using
perceptual hashing (pHash) + Hamming distance — the second half of the
1vcian approach:

> "Once potential card candidates were extracted, I used a Hamming distance
>  algorithm on perceptual hashes of the images to compare them against
>  precomputed hashes from the card database."

**How it works:**
1. Pre-compute a 64-bit pHash for every card in `datasets/` (fast, ~1ms each)
2. For a query crop, compute its pHash
3. Find the database card with the minimum Hamming distance (# of differing bits)
4. Accept match if distance ≤ threshold (typically 10–15 out of 64 bits)

pHash is robust to JPEG compression, minor scaling, and colour shifts —
exactly the distortions that happen during phone photography.

**Install:** `pip install imagehash`

In [ ]:
# ── Centering helpers (Python port of our cvDetectors.ts logic) ───────────────
def detect_card_bounds_cv(img_bgr: np.ndarray) -> Optional[dict]:
    """
    Find the outer card boundary in a crop using Sobel gradient.
    Returns {x, y, w, h} or None.
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    sobel_x = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sobel_x**2 + sobel_y**2)

    # Column profile for left/right edges
    col_profile = mag.mean(axis=0)
    threshold = col_profile.max() * 0.3
    left  = next((i for i in range(w) if col_profile[i] > threshold), 0)
    right = next((i for i in range(w-1, -1, -1) if col_profile[i] > threshold), w-1)

    # Row profile for top/bottom edges
    row_profile = mag.mean(axis=1)
    top    = next((i for i in range(h) if row_profile[i] > threshold), 0)
    bottom = next((i for i in range(h-1, -1, -1) if row_profile[i] > threshold), h-1)

    if right <= left or bottom <= top:
        return None
    return {"x": left, "y": top, "w": right - left, "h": bottom - top}


def detect_inner_frame_cv(card_bgr: np.ndarray) -> Optional[dict]:
    """
    Scan inward from each edge using a Sobel gradient profile to find the
    inner printed frame boundary. Returns {x, y, w, h, confidence} or None.
    """
    gray = cv2.cvtColor(card_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    sobel_x = np.abs(cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3))
    sobel_y = np.abs(cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3))

    SCAN_FRAC = 0.30
    PEAK_RATIO = 0.25

    def find_edge_from(profile, peak_ratio=PEAK_RATIO):
        threshold = profile.max() * peak_ratio
        idx = np.where(profile > threshold)[0]
        return int(idx[0]) if len(idx) > 0 else None

    left_scan  = int(w * SCAN_FRAC)
    right_scan = int(w * SCAN_FRAC)
    top_scan   = int(h * SCAN_FRAC)
    bot_scan   = int(h * SCAN_FRAC)

    col_l = sobel_x[:, :left_scan].mean(axis=0)
    col_r = sobel_x[:, w-right_scan:].mean(axis=0)[::-1]
    row_t = sobel_y[:top_scan, :].mean(axis=1)
    row_b = sobel_y[h-bot_scan:, :].mean(axis=1)[::-1]

    frame_left   = find_edge_from(col_l)
    frame_right  = find_edge_from(col_r)
    frame_top    = find_edge_from(row_t)
    frame_bottom = find_edge_from(row_b)

    if any(v is None for v in [frame_left, frame_right, frame_top, frame_bottom]):
        return None

    frame_x  = frame_left
    frame_y  = frame_top
    frame_w  = w - frame_left - frame_right
    frame_h  = h - frame_top  - frame_bottom

    if frame_w <= 0 or frame_h <= 0:
        return None

    margin_symmetry = 1 - abs(frame_left - frame_right) / max(frame_left + frame_right, 1)
    confidence = "high" if margin_symmetry > 0.7 else "medium" if margin_symmetry > 0.4 else "low"

    return {"x": frame_x, "y": frame_y, "w": frame_w, "h": frame_h,
            "confidence": confidence,
            "margins": {"left": frame_left, "right": frame_right,
                        "top": frame_top, "bottom": frame_bottom}}


def build_centering(card_w, card_h, inner) -> dict:
    """
    Compute centering ratios from outer card dims + inner frame margins.
    All margins expressed as % of card dimension (sum to 100).
    """
    m = inner["margins"]
    total_h = m["left"] + m["right"]
    total_v = m["top"]  + m["bottom"]

    if total_h == 0 or total_v == 0:
        return {}

    left_pct   = round(m["left"]   / total_h * 100, 1)
    right_pct  = round(m["right"]  / total_h * 100, 1)
    top_pct    = round(m["top"]    / total_v * 100, 1)
    bottom_pct = round(m["bottom"] / total_v * 100, 1)

    lr_ratio = max(left_pct, right_pct) / min(left_pct, right_pct) if min(left_pct, right_pct) > 0 else 99
    tb_ratio = max(top_pct, bottom_pct) / min(top_pct, bottom_pct) if min(top_pct, bottom_pct) > 0 else 99

    def interpret(ratio):
        if ratio <= 1.1:  return "well_centered"
        if ratio <= 1.35: return "slightly_off"
        if ratio <= 1.65: return "noticeably_off"
        return "severely_off"

    return {
        "left": left_pct, "right": right_pct,
        "top": top_pct,   "bottom": bottom_pct,
        "lr_ratio": round(lr_ratio, 2), "tb_ratio": round(tb_ratio, 2),
        "lr_interp": interpret(lr_ratio),
        "tb_interp": interpret(tb_ratio),
    }


def plot_centering_on_crop(crop_bgr: np.ndarray, title: str = ""):
    """Visualise outer border + inner frame + centering ratios on a warped crop."""
    h, w = crop_bgr.shape[:2]

    inner = detect_inner_frame_cv(crop_bgr)

    fig, ax = plt.subplots(figsize=(5, 7))
    ax.imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))

    # Outer card boundary (blue)
    outer_rect = mpatches.Rectangle((0, 0), w-1, h-1,
                                    linewidth=2, edgecolor="royalblue",
                                    facecolor="none", linestyle="--")
    ax.add_patch(outer_rect)

    centering = {}
    if inner:
        # Inner frame (yellow)
        inner_rect = mpatches.Rectangle(
            (inner["x"], inner["y"]), inner["w"], inner["h"],
            linewidth=2, edgecolor="gold", facecolor="none"
        )
        ax.add_patch(inner_rect)
        centering = build_centering(w, h, inner)

        if centering:
            ax.text(w/2, h + 20,
                    f"L/R: {centering['left']:.0f}/{centering['right']:.0f}  "
                    f"T/B: {centering['top']:.0f}/{centering['bottom']:.0f}  "
                    f"[{centering['lr_interp']}]",
                    ha="center", va="top", fontsize=8,
                    transform=ax.transData)

    ax.set_title(f"{title}\nInner frame: {'found' if inner else 'not found'}", fontsize=9)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    return centering


# Run centering on all saved crops
print("Centering Analysis on OBB Crops")
print("═" * 60)
for crop_path in saved:
    crop_bgr = cv2.imread(str(crop_path))
    if crop_bgr is None:
        continue
    c = plot_centering_on_crop(crop_bgr, title=crop_path.stem)
    if c:
        print(f"  {crop_path.stem}: L/R {c['left']:.0f}/{c['right']:.0f} ({c['lr_interp']}) "
              f"| T/B {c['top']:.0f}/{c['bottom']:.0f} ({c['tb_interp']})")

---
## Part B — Custom Training Dataset
### B1 — Gather source card images

In [ ]:
# ── B1 — Download CLEAN official card images for compositing ───────────────────
# The local datasets are slab thumbnails / eBay scene photos, NOT clean card
# cutouts, so compositing them would produce WRONG OBB labels. Instead we pull
# clean official card faces (~600x825, ~1.4 aspect) from the Pokemon TCG CDN
# listed in pokemon-cards.csv. Cached to CARDS_DIR so re-runs are instant.
import csv, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

TCG_CSV = Path("datasets/pokemon-cards.csv")
_UA = {"User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) Safari/537.36")}
CARDS_DIR.mkdir(parents=True, exist_ok=True)

# (id, url) pairs from the CSV, random subset of N_TCG_CARDS
rows = []
with open(TCG_CSV, newline="") as f:
    for r in csv.DictReader(f):
        u = r.get("image_url", "")
        if u.startswith("http"):
            rows.append((r["id"], u))
random.shuffle(rows)
rows = rows[:N_TCG_CARDS]

def _fetch(idu):
    cid, url = idu
    out = CARDS_DIR / f"{cid}.png"
    if out.exists() and out.stat().st_size > 1024:
        return True                                  # cached
    try:
        req = urllib.request.Request(url, headers=_UA)
        out.write_bytes(urllib.request.urlopen(req, timeout=20).read())
        return True
    except Exception:
        return False

ok = 0
with ThreadPoolExecutor(max_workers=12) as ex:
    futs = [ex.submit(_fetch, idu) for idu in rows]
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Downloading clean cards"):
        ok += bool(fut.result())

card_files = sorted(p for p in CARDS_DIR.glob("*.png") if p.stat().st_size > 1024)
print(f"\nClean card images ready: {len(card_files)}  (ok {ok}/{len(rows)} this run)")

# Preview a few
sample = random.sample(card_files, min(6, len(card_files)))
fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 4))
for ax, p in zip(np.atleast_1d(axes).ravel(), sample):
    try:
        im = Image.open(p).convert("RGB"); im.thumbnail((200, 280)); ax.imshow(im)
        ax.set_title(p.stem[:14], fontsize=7)
    except Exception:
        ax.set_title("err")
    ax.axis("off")
plt.suptitle(f"Clean compositing cards ({len(card_files)} total)", fontsize=10)
plt.tight_layout(); plt.show()

### B2 — Generate background images

If you don't have a `datasets/backgrounds/` folder, this cell creates simple
synthetic backgrounds (solid colours, gradients, noise textures).
For better results: add real photos of tables, hands, shelves, etc.

In [ ]:
# ── B2 — Generate diverse + HARD backgrounds ──────────────────────────────────
# Solid/gradient/noise alone won't teach the model the conditions that break
# classical CV: DARK surfaces (low edge contrast) and BUSY FABRIC. We synthesise
# a harder mix (dark & fabric weighted up). Drop real photos (tables, desks,
# mats, hands) into BACKGROUNDS_DIR and they are used automatically too.
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
BACKGROUNDS_DIR.mkdir(parents=True, exist_ok=True)
existing_bgs = [p for p in BACKGROUNDS_DIR.glob("*.*") if p.suffix.lower() in IMG_EXTS]
print(f"Backgrounds already present: {len(existing_bgs)}")

N_SYNTHETIC_BGS = 160
H_, W_ = 640, 640

def _bg_solid():
    return np.full((H_, W_, 3), [random.randint(20, 235) for _ in range(3)], np.uint8)

def _bg_dark():                                    # dark surface = the low-contrast case
    base = np.array([random.randint(8, 45) for _ in range(3)])
    arr = np.clip(base + np.random.randint(-8, 8, (H_, W_, 3)), 0, 255).astype(np.uint8)
    return cv2.GaussianBlur(arr, (7, 7), 0)

def _bg_gradient():
    c1 = np.array([random.randint(15, 210) for _ in range(3)], float)
    c2 = np.array([random.randint(15, 210) for _ in range(3)], float)
    t = np.linspace(0, 1, H_)[:, None, None]
    return (c1 + (c2 - c1) * t).astype(np.uint8).repeat(W_, axis=1)

def _bg_noise():
    lo = random.randint(60, 130)
    arr = np.random.randint(lo, min(lo + 80, 255), (H_, W_, 3), np.uint8)
    return cv2.GaussianBlur(arr, (random.choice([15, 21, 31]),) * 2, 0)

def _bg_wood():
    base = np.array([random.randint(70, 165), random.randint(45, 110), random.randint(20, 70)], float)
    rows = base + np.random.randint(-18, 18, (H_, 1, 3))
    arr = np.clip(rows.repeat(W_, axis=1), 0, 255).astype(np.uint8)
    return cv2.GaussianBlur(arr, (1, 9), 0)        # grain along one axis

def _bg_fabric():                                  # busy fabric w/ swirls (pink-fabric case)
    base = np.array([random.randint(120, 230) for _ in range(3)], float)
    arr = np.clip(base + np.random.randint(-25, 25, (H_, W_, 3)), 0, 255).astype(np.uint8)
    arr = cv2.GaussianBlur(arr, (9, 9), 0)
    for _ in range(random.randint(3, 7)):
        c = (random.randint(0, W_), random.randint(0, H_))
        col = tuple(int(x) for x in np.clip(base + np.random.randint(-60, 60, 3), 0, 255))
        cv2.ellipse(arr, c, (random.randint(60, 260), random.randint(60, 260)),
                    random.randint(0, 180), 0, random.randint(120, 360), col, random.randint(6, 22))
    return cv2.GaussianBlur(arr, (7, 7), 0)

def _vignette(arr):                                # uneven lighting
    yy, xx = np.mgrid[0:H_, 0:W_]
    cx, cy = random.uniform(0.3, 0.7) * W_, random.uniform(0.3, 0.7) * H_
    d = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    m = 1 - (d / d.max()) * random.uniform(0.25, 0.6)
    return np.clip(arr * m[..., None], 0, 255).astype(np.uint8)

_gens = [_bg_solid, _bg_dark, _bg_dark, _bg_gradient, _bg_noise,
         _bg_wood, _bg_fabric, _bg_fabric]         # dark + fabric weighted up

if len(list(BACKGROUNDS_DIR.glob("bg_*.jpg"))) < N_SYNTHETIC_BGS:
    print("Generating synthetic backgrounds (dark/fabric/wood/gradient/noise)...")
    for i in range(N_SYNTHETIC_BGS):
        arr = random.choice(_gens)()
        if random.random() < 0.5:
            arr = _vignette(arr)
        cv2.imwrite(str(BACKGROUNDS_DIR / f"bg_{i:04d}.jpg"), arr, [cv2.IMWRITE_JPEG_QUALITY, 88])
    print(f"Generated {N_SYNTHETIC_BGS} backgrounds → {BACKGROUNDS_DIR}")

bg_files = [p for p in BACKGROUNDS_DIR.glob("*.*") if p.suffix.lower() in IMG_EXTS]
print(f"Total backgrounds: {len(bg_files)}")

### B3 — Synthetic composite generator

Adapted directly from `trainCreator.py` (1vcian/Pokemon-TCGP-Card-Scanner).
Key changes for our use case:
- Works with JPEG/WebP card images (not only PNG with alpha)
- Larger scale range (10–85%) to cover eBay-style single-card shots
- Extra placement mode: single large card (eBay listing simulation)
- White border synthesis to simulate PSA/raw card white edges

In [ ]:
# ── IoU helper ────────────────────────────────────────────────────────────────
def iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    u = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
    return inter / u if u > 0 else 0


# ── Single card placement ─────────────────────────────────────────────────────
def place_card(card_pil: Image.Image, bg_pil: Image.Image,
               existing_boxes: list, scale: Optional[float] = None,
               max_attempts: int = 50):
    """
    Composite `card_pil` onto `bg_pil` at a random position.
    Returns (composited_image, yolo_obb_label_list, pixel_bbox) or None on failure.

    YOLO OBB label format: [class, x1,y1, x2,y2, x3,y3, x4,y4]  (0-1 normalised)
    """
    bg_w, bg_h = bg_pil.size
    card_w, card_h = card_pil.size

    if scale is None:
        scale = random.uniform(0.10, 0.85)

    new_w = int(bg_w * scale)
    new_h = int(card_h * new_w / card_w)
    orig_w, orig_h = new_w, new_h
    card_resized = card_pil.resize((new_w, new_h), Image.LANCZOS)

    # Rotation: ±5° for large cards, ±20° for small
    max_angle = 5 if scale > 0.4 else 20
    angle = random.uniform(-max_angle, max_angle)
    card_rotated = card_resized.rotate(angle, expand=True, resample=Image.BICUBIC)
    rot_w, rot_h = card_rotated.size

    paste_x, paste_y = 0, 0
    for _ in range(max_attempts):
        px = random.randint(0, max(0, bg_w - rot_w))
        py = random.randint(0, max(0, bg_h - rot_h))
        x1, y1 = px, py
        x2, y2 = px + rot_w, py + rot_h
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(bg_w, x2), min(bg_h, y2)
        if (x2 - x1) < 20 or (y2 - y1) < 20:
            continue
        pixel_bbox = [x1, y1, x2, y2]
        if any(iou(pixel_bbox, eb) > 0.1 for eb in existing_boxes):
            continue
        paste_x, paste_y = px, py
        break
    else:
        return None   # couldn't place without overlap

    # Composite
    result = bg_pil.copy()
    if card_rotated.mode == "RGBA":
        mask = card_rotated.split()[3]
        result.paste(card_rotated.convert("RGB"), (paste_x, paste_y), mask)
    else:
        result.paste(card_rotated, (paste_x, paste_y))

    # OBB corner calculation (rotate unrotated corners around centre)
    cx = paste_x + rot_w / 2
    cy = paste_y + rot_h / 2
    hw, hh = orig_w / 2, orig_h / 2
    raw_corners = [(-hw, -hh), (hw, -hh), (hw, hh), (-hw, hh)]
    rad = math.radians(-angle)   # PIL rotates CCW → negate
    obb_coords = [0]  # class 0 = pokemon_card
    for (rx, ry) in raw_corners:
        xr = rx * math.cos(rad) - ry * math.sin(rad)
        yr = rx * math.sin(rad) + ry * math.cos(rad)
        obb_coords.append(min(1, max(0, (cx + xr) / bg_w)))
        obb_coords.append(min(1, max(0, (cy + yr) / bg_h)))

    return result, obb_coords, pixel_bbox


def draw_card_stand(img_pil, pixel_bbox):
    """Overlay a translucent display-stand (two clear-plastic legs + base) on the
    LOWER part of a card — the common collector/eBay photo that broke classical CV.
    The OBB LABEL IS NOT CHANGED: the card's true corners stay the target, so the
    model learns the card boundary despite a stand/occlusion below it.
    Returns a new PIL image."""
    x1, y1, x2, y2 = [int(v) for v in pixel_bbox]
    w, h = x2 - x1, y2 - y1
    if w < 30 or h < 30:
        return img_pil
    arr = np.array(img_pil.convert("RGB"))
    Hh, Ww = arr.shape[:2]
    overlay = arr.copy()
    cx = (x1 + x2) // 2
    base_y = min(Hh - 1, int(y2 + h * random.uniform(0.02, 0.10)))   # just below the card
    top_y  = int(y2 - h * random.uniform(0.10, 0.25))                # legs reach onto card
    spread = int(w * random.uniform(0.18, 0.34))
    leg_w  = max(3, int(w * random.uniform(0.03, 0.06)))
    tint   = random.randint(170, 230)                                # light translucent plastic
    for dx in (-spread, spread):
        pts = np.array([[cx + dx - leg_w, top_y],         [cx + dx + leg_w, top_y],
                        [cx + dx // 2 + leg_w, base_y],    [cx + dx // 2 - leg_w, base_y]], np.int32)
        cv2.fillConvexPoly(overlay, pts, (tint, tint, tint))
    cv2.rectangle(overlay, (cx - spread - leg_w, base_y - leg_w),
                  (cx + spread + leg_w, base_y + leg_w), (tint, tint, tint), -1)
    alpha = random.uniform(0.35, 0.6)                                # translucency
    out = cv2.addWeighted(overlay, alpha, arr, 1 - alpha, 0)
    return Image.fromarray(out)


print("Composite helpers defined (place_card + draw_card_stand).")

### B4 — Dataset generator loop

In [ ]:
def generate_dataset(
    card_files: list,
    bg_files:   list,
    out_dir:    Path,
    n_images:   int = N_TRAIN_IMAGES,
    img_size:   int = 640,
    train_frac: float = 0.8,
    val_frac:   float = 0.1,
    stand_prob: float = 0.5,
) -> Path:
    """
    Generate a YOLO OBB dataset with `n_images` synthetic composites.

    Three scene modes (sampled randomly):
      'single_large'  : 1 card at 50–85% width  (eBay-style clean listing)
      'few_cards'     : 2–4 cards at 20–40% width
      'many_cards'    : 5–15 cards at 8–18% width

    Returns path to data.yaml.
    """
    splits = {
        "train": int(n_images * train_frac),
        "val":   int(n_images * val_frac),
        "test":  n_images - int(n_images * train_frac) - int(n_images * val_frac),
    }

    for split in splits:
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    idx = 0
    for split, count in splits.items():
        imgs_dir   = out_dir / "images" / split
        labels_dir = out_dir / "labels" / split

        for _ in tqdm(range(count), desc=f"Generating {split}"):
            bg_path = random.choice(bg_files)
            bg_pil  = Image.open(bg_path).convert("RGB")
            bg_pil  = bg_pil.resize((img_size, img_size), Image.LANCZOS)

            # Choose scene mode
            mode = random.choices(
                ["single_large", "few_cards", "many_cards"],
                weights=[0.40, 0.35, 0.25]
            )[0]

            if mode == "single_large":
                n_cards = 1
                scale_range = (0.50, 0.85)
            elif mode == "few_cards":
                n_cards = random.randint(2, 4)
                scale_range = (0.20, 0.40)
            else:
                n_cards = random.randint(5, 15)
                scale_range = (0.08, 0.18)

            existing_boxes = []
            labels = []
            composite = bg_pil

            for _ in range(n_cards):
                card_path = random.choice(card_files)
                try:
                    card_pil = Image.open(card_path).convert("RGBA")
                except Exception:
                    continue
                scale = random.uniform(*scale_range)
                result = place_card(card_pil, composite, existing_boxes, scale)
                if result is None:
                    continue
                composite, obb_label, pixel_bbox = result
                existing_boxes.append(pixel_bbox)
                labels.append(" ".join(f"{v:.6f}" if i > 0 else str(int(v))
                                       for i, v in enumerate(obb_label)))

            if not labels:
                continue   # skip empty images

            # Stand/occlusion augmentation for upright single cards: simulate the
            # collector photo (card on a clear stand). Label stays on the true
            # card corners, so the model learns the boundary despite the stand.
            if mode == "single_large" and existing_boxes and random.random() < stand_prob:
                composite = draw_card_stand(composite, existing_boxes[0])

            img_name   = f"img_{idx:07d}.jpg"
            label_name = f"img_{idx:07d}.txt"
            composite.save(imgs_dir / img_name, quality=90)
            (labels_dir / label_name).write_text("\n".join(labels))
            idx += 1

    # Write data.yaml
    yaml_data = {
        "path":   str(out_dir.resolve()),
        "train":  "images/train",
        "val":    "images/val",
        "test":   "images/test",
        "nc":     1,
        "names":  ["pokemon_card"],
    }
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(yaml.dump(yaml_data, default_flow_style=False))
    print(f"\n✅  Dataset generated: {idx} images → {out_dir}")
    print(f"   data.yaml: {yaml_path}")
    return yaml_path


# ── Run dataset generation ─────────────────────────────────────────────────────
# Set N_TRAIN_IMAGES in the config cell. Start small (200–500) to verify
# quality, then increase to 5000–10000 for production training.
yaml_path = generate_dataset(card_files, bg_files, DATASET_OUT, n_images=N_TRAIN_IMAGES)

### B5 — Preview synthetic composites

In [ ]:
# Show 6 random generated training images with OBB labels drawn
train_imgs = list((DATASET_OUT / "images" / "train").glob("*.jpg"))
sample_imgs = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    img_bgr = cv2.imread(str(img_path))
    label_path = DATASET_OUT / "labels" / "train" / (img_path.stem + ".txt")

    if label_path.exists():
        h, w = img_bgr.shape[:2]
        for line in label_path.read_text().strip().split("\n"):
            vals = list(map(float, line.split()))
            # vals[1:] = x1,y1,x2,y2,x3,y3,x4,y4 normalised
            pts = np.array(vals[1:]).reshape(4, 2)
            pts[:, 0] *= w
            pts[:, 1] *= h
            pts = pts.astype(np.int32)
            cv2.polylines(img_bgr, [pts], isClosed=True,
                          color=(0, 255, 255), thickness=2)

    ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis("off")

plt.suptitle("Sample synthetic training images with OBB labels", fontsize=12)
plt.tight_layout()
plt.show()

### B6 — Train YOLO11n-obb on generated dataset

In [ ]:
# ── Training config ───────────────────────────────────────────────────────────
EPOCHS      = 50      # increase to 100-150 for full training
IMG_SZ      = 640
BATCH       = 8       # reduce to 4 if OOM on MPS
DEVICE      = "mps"   # Apple Silicon; use 'cuda' for NVIDIA, 'cpu' as fallback

# ─────────────────────────────────────────────────────────────────────────────
print(f"Training YOLO11n-obb for {EPOCHS} epochs on {yaml_path}")
print(f"Device: {DEVICE}  |  Batch: {BATCH}  |  ImgSz: {IMG_SZ}")

train_model = YOLO(PRETRAINED_BASE)   # YOLO11n-obb as starting point

results = train_model.train(
    data    = str(yaml_path),
    epochs  = EPOCHS,
    imgsz   = IMG_SZ,
    batch   = BATCH,
    device  = DEVICE,
    project = str(DATASET_OUT),
    name    = "train",
    exist_ok= True,
    patience= 20,          # early stop if no improvement for 20 epochs
    save    = True,
    plots   = True,
    cache   = True,        # cache images in RAM for faster epochs
    # ── Augmentation ──────────────────────────────────────────────────────────
    # Our synthetic composites are rotation-only; add PERSPECTIVE + geometric/
    # photometric jitter here so the model generalises to tilted, real photos.
    degrees     = 15.0,    # extra in-plane rotation
    perspective = 0.0008,  # mild homography  -> trapezoidal cards
    scale       = 0.4,     # zoom in/out
    translate   = 0.1,
    fliplr      = 0.5,
    flipud      = 0.0,     # cards are rarely upside-down in listings
    hsv_h       = 0.015,
    hsv_s       = 0.5,     # colour/lighting variation
    hsv_v       = 0.4,     # brightness (helps dark-background cases)
    mosaic      = 1.0,
    close_mosaic= 10,      # disable mosaic for the last 10 epochs
)

# train() returns metrics (list); save_dir lives on the trainer object
best_weights = Path(train_model.trainer.save_dir) / "weights" / "best.pt"
# Fallback: also check the path we specified, in case it resolved locally
_fallback = DATASET_OUT / "train" / "weights" / "best.pt"
if not best_weights.exists() and _fallback.exists():
    best_weights = _fallback
print(f"\nTraining complete.")
print(f"  save_dir  : {train_model.trainer.save_dir}")
print(f"  best.pt   : {best_weights}")
print(f"  exists    : {best_weights.exists()}")

### B7 — Evaluate custom model & compare with base

In [ ]:
# ── Resolve best_weights robustly (handles cross-session restarts) ─────────────
if "best_weights" not in dir() or not Path(str(best_weights)).exists():
    import glob as _glob
    # ultralytics may write to /opt/homebrew/runs, ~/runs, or the project dir
    _search_roots = [Path.home(), Path("/opt/homebrew/runs"), Path("/opt/homebrew")]
    _patterns = [str(r / "**" / "yolo_obb_cards" / "train*" / "weights" / "best.pt") for r in _search_roots]
    _patterns += [str(DATASET_OUT / "train*" / "weights" / "best.pt")]
    _candidates = sorted(
        [p for pat in _patterns for p in _glob.glob(pat, recursive=True)],
        key=lambda p: Path(p).stat().st_mtime,
        reverse=True,
    )
    if _candidates:
        best_weights = Path(_candidates[0])
        print(f"Found weights: {best_weights}")
    else:
        best_weights = DATASET_OUT / "train" / "weights" / "best.pt"  # will fail gracefully below
# ──────────────────────────────────────────────────────────────────────────────

if best_weights.exists():
    custom_model = YOLO(str(best_weights))

    print("Custom model — test on real card photos")
    print("═" * 60)
    for img_path in TEST_IMAGES:
        if not Path(img_path).exists():
            continue
        img_bgr = cv2.imread(img_path)
        results = custom_model.predict(img_bgr, conf=CONF_THRESHOLD, verbose=False)
        n_det   = len(results[0].obb) if results[0].obb is not None else 0

        fig, ax = plt.subplots(figsize=(6, 8))
        vis = img_bgr.copy()
        if results[0].obb is not None:
            for i in range(n_det):
                corners = results[0].obb.xyxyxyxy[i].cpu().numpy()
                score   = float(results[0].obb.conf[i].cpu())
                pts     = order_corners(corners).astype(np.int32)
                cv2.polylines(vis, [pts], True, (0, 255, 0), 3)
                cv2.putText(vis, f"{score:.2f}", tuple(pts[0]),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{Path(img_path).name}  |  {n_det} detected", fontsize=9)
        ax.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("No trained weights found — run B6 first.")

### B8 — Full pipeline: noisy photo → warped crop → centering

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
PIPELINE_IMAGE = "cards/image0_front.jpeg"   # ← any photo with a Pokemon card
# ──────────────────────────────────────────────────────────────────────────────

# Use best available model
pipeline_model = YOLO(str(best_weights)) if best_weights.exists() else model

img_bgr  = cv2.imread(PIPELINE_IMAGE)
results  = pipeline_model.predict(img_bgr, conf=CONF_THRESHOLD, verbose=False)
obb_res  = results[0].obb

if obb_res is None or len(obb_res) == 0:
    print("No cards detected. Try lowering CONF_THRESHOLD or use a trained model.")
else:
    # Take highest-confidence detection
    best_i   = int(obb_res.conf.argmax())
    corners  = obb_res.xyxyxyxy[best_i].cpu().numpy()
    score    = float(obb_res.conf[best_i].cpu())
    crop_bgr = perspective_warp(img_bgr, corners)

    # Centering on warped crop
    inner    = detect_inner_frame_cv(crop_bgr)
    centering = build_centering(CARD_W, CARD_H, inner) if inner else {}

    # Visualise 3-stage pipeline
    fig, axes = plt.subplots(1, 3, figsize=(16, 6))

    # Stage 1: detection
    vis = img_bgr.copy()
    pts = order_corners(corners).astype(np.int32)
    cv2.polylines(vis, [pts], True, (0, 255, 255), 3)
    axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"1. OBB Detection (conf={score:.2f})", fontsize=10)
    axes[0].axis("off")

    # Stage 2: warped crop
    axes[1].imshow(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"2. Perspective Warp ({CARD_W}×{CARD_H}px)", fontsize=10)
    axes[1].axis("off")

    # Stage 3: centering overlay
    crop_vis = crop_bgr.copy()
    h, w = crop_vis.shape[:2]
    if inner:
        # Draw outer (blue dashed) + inner (yellow)
        cv2.rectangle(crop_vis, (0,0), (w-1,h-1), (255, 80, 0), 2)
        cv2.rectangle(crop_vis,
                      (inner["x"], inner["y"]),
                      (inner["x"]+inner["w"], inner["y"]+inner["h"]),
                      (0, 215, 255), 2)
    axes[2].imshow(cv2.cvtColor(crop_vis, cv2.COLOR_BGR2RGB))
    if centering:
        title = (f"3. Centering\n"
                 f"L/R {centering['left']:.0f}/{centering['right']:.0f} "
                 f"({centering['lr_interp']})\n"
                 f"T/B {centering['top']:.0f}/{centering['bottom']:.0f} "
                 f"({centering['tb_interp']})")
    else:
        title = "3. Centering\n(inner frame not found)"
    axes[2].set_title(title, fontsize=10)
    axes[2].axis("off")

    plt.suptitle("Full Pipeline: Photo → OBB → Warp → Centering", fontsize=13)
    plt.tight_layout()
    plt.show()

    print("\nCentering result:")
    for k, v in centering.items():
        print(f"  {k}: {v}")